# Task 2: Dataset Acquisition and Understanding
**Singularity Research Lab – AI Division**  
**Project:** Electricity Bill Predictor  
**Dataset:** UCI Individual Household Electric Power Consumption (`household_power_consumption.txt`)

---

## 1. Objective and Scope
The goal of this notebook is to perform initial data acquisition, ingestion, and structural inspection of the raw smart meter dataset.
This inspection establishes the foundational understanding of:
- High-frequency electric telemetry measurements.
- Exact dataset dimensions, column data types, and delimiters.
- Missing value mechanisms and integrity constraints.
- Temporal resolution, timestamp formats, and sampling frequency.
- The physics and domain distinctions between instantaneous power (kW) and energy consumption (Wh/kWh).


In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)


## 2. Ingestion of Raw Dataset
We ingest the raw dataset from `data/household_power_consumption.txt`.
Key considerations:
- The dataset utilizes a semicolon (`;`) delimiter.
- Missing values are explicitly encoded as question marks (`?`).
- The raw dataset is kept completely immutable.


In [2]:
data_path = os.path.join('..', 'data', 'household_power_consumption.txt')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'household_power_consumption.txt')

print(f"Loading raw data from: {data_path}")
df = pd.read_csv(
    data_path,
    sep=';',
    na_values='?',
    low_memory=False
)
print("Data loaded successfully.")


Loading raw data from: data/household_power_consumption.txt
Data loaded successfully.


## 3. Dataset Dimensions and Structural Schema
Let us examine the total record count, column names, memory footprint, and assigned Pandas dtypes.


In [3]:
print(f"Total Rows (Observations): {df.shape[0]:,}")
print(f"Total Columns (Features):    {df.shape[1]}")
print("\n--- Column Names & Data Types ---")
df_info = pd.DataFrame({
    'Data Type': df.dtypes.astype(str),
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum()
})
print(df_info.to_string())


Total Rows (Observations): 2,075,259
Total Columns (Features):    9

--- Column Names & Data Types ---
                      Data Type  Non-Null Count  Null Count
Date                     object         2075259           0
Time                     object         2075259           0
Global_active_power     float64         2049280       25979
Global_reactive_power   float64         2049280       25979
Voltage                 float64         2049280       25979
Global_intensity        float64         2049280       25979
Sub_metering_1          float64         2049280       25979
Sub_metering_2          float64         2049280       25979
Sub_metering_3          float64         2049280       25979


## 4. First and Last Five Observations
Examining the boundary records confirms the chronological sequence, time formats, and sub-metering readings.


In [4]:
print("--- First 5 Rows (Head) ---")
print(df.head(5).to_string())

print("\n--- Last 5 Rows (Tail) ---")
print(df.tail(5).to_string())


--- First 5 Rows (Head) ---
         Date      Time  Global_active_power  Global_reactive_power  Voltage  Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3
0  16/12/2006  17:24:00               4.2160                 0.4180 234.8400           18.4000          0.0000          1.0000         17.0000
1  16/12/2006  17:25:00               5.3600                 0.4360 233.6300           23.0000          0.0000          1.0000         16.0000
2  16/12/2006  17:26:00               5.3740                 0.4980 233.2900           23.0000          0.0000          2.0000         17.0000
3  16/12/2006  17:27:00               5.3880                 0.5020 233.7400           23.0000          0.0000          1.0000         17.0000
4  16/12/2006  17:28:00               3.6660                 0.5280 235.6800           15.8000          0.0000          1.0000         17.0000

--- Last 5 Rows (Tail) ---
               Date      Time  Global_active_power  Global_reactive_power  Voltage  Gl

## 5. Missing Value Analysis
Missing telemetry values indicate sensor outages, transmission failures, or smart meter disconnections.
Here we quantify exact counts and percentages across all variables.


In [5]:
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_table = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage (%)': missing_pct
})
print("--- Missing Values by Column ---")
print(missing_table.to_string())

numerical_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                  'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

all_null_rows = int(df[numerical_cols].isnull().all(axis=1).sum())
any_null_rows = int(df[numerical_cols].isnull().any(axis=1).sum())

print(f"\nRows where ALL numerical power readings are missing: {all_null_rows:,}")
print(f"Rows where ANY numerical power reading is missing:  {any_null_rows:,}")
print(f"Synchronous Missing Ratio: {all_null_rows / any_null_rows:.1%}")


--- Missing Values by Column ---
                       Missing Count  Missing Percentage (%)
Date                               0                  0.0000
Time                               0                  0.0000
Global_active_power            25979                  1.2518
Global_reactive_power          25979                  1.2518
Voltage                        25979                  1.2518
Global_intensity               25979                  1.2518
Sub_metering_1                 25979                  1.2518
Sub_metering_2                 25979                  1.2518
Sub_metering_3                 25979                  1.2518

Rows where ALL numerical power readings are missing: 25,979
Rows where ANY numerical power reading is missing:  25,979
Synchronous Missing Ratio: 100.0%


## 6. Examination of Date and Time Formats
The temporal resolution is minute-by-minute. We verify the formatting conventions of `Date` and `Time` strings.


In [6]:
print("Date Format Sample:", df['Date'].iloc[0], "(Day/Month/Year format: DD/MM/YYYY)")
print("Time Format Sample:", df['Time'].iloc[0], "(24-Hour format: HH:MM:SS)")

start_str = f"{df['Date'].iloc[0]} {df['Time'].iloc[0]}"
end_str = f"{df['Date'].iloc[-1]} {df['Time'].iloc[-1]}"
print(f"Dataset Temporal Span: {start_str}  -->  {end_str}")


Date Format Sample: 16/12/2006 (Day/Month/Year format: DD/MM/YYYY)
Time Format Sample: 17:24:00 (24-Hour format: HH:MM:SS)
Dataset Temporal Span: 16/12/2006 17:24:00  -->  26/11/2010 21:02:00


## 7. Descriptive Summary Statistics
Summary statistics provide an overview of the range, mean, and dispersion of electrical consumption metrics.


In [7]:
print("--- Descriptive Summary Statistics ---")
print(df.describe().T.to_string())


--- Descriptive Summary Statistics ---
                             count     mean    std      min      25%      50%      75%      max
Global_active_power   2049280.0000   1.0916 1.0573   0.0760   0.3080   0.6020   1.5280  11.1220
Global_reactive_power 2049280.0000   0.1237 0.1127   0.0000   0.0480   0.1000   0.1940   1.3900
Voltage               2049280.0000 240.8399 3.2400 223.2000 238.9900 241.0100 242.8900 254.1500
Global_intensity      2049280.0000   4.6278 4.4444   0.2000   1.4000   2.6000   6.4000  48.4000
Sub_metering_1        2049280.0000   1.1219 6.1530   0.0000   0.0000   0.0000   0.0000  88.0000
Sub_metering_2        2049280.0000   1.2985 5.8220   0.0000   0.0000   0.0000   1.0000  80.0000
Sub_metering_3        2049280.0000   6.4584 8.4372   0.0000   0.0000   1.0000  17.0000  31.0000


## 8. Technical Domain Explanations

### A. Column Descriptions and Measurement Units
1. **`Date`**: Date of measurement in `dd/mm/yyyy` format (spanning 16/12/2006 to 26/11/2010).
2. **`Time`**: Time of measurement in `hh:mm:ss` format (recorded at 1-minute frequency).
3. **`Global_active_power`**: Household global minute-averaged active power in **kilowatts (kW)**.
4. **`Global_reactive_power`**: Household global minute-averaged reactive power in **kilowatts (kW)** (physically, reactive power is volt-ampere reactive / kVAR, but reported as kW in the dataset documentation).
5. **`Voltage`**: Minute-averaged line voltage in **volts (V)**.
6. **`Global_intensity`**: Household global minute-averaged electrical current in **amperes (A)**.
7. **`Sub_metering_1`**: Active energy in **watt-hours (Wh)** for Sub-meter 1 (Kitchen appliances: dishwasher, microwave, oven).
8. **`Sub_metering_2`**: Active energy in **watt-hours (Wh)** for Sub-meter 2 (Laundry room: washing machine, tumble dryer, refrigerator, light).
9. **`Sub_metering_3`**: Active energy in **watt-hours (Wh)** for Sub-meter 3 (Electric water heater and air conditioner).

---

### B. Active Power vs. Energy Consumption
- **Active Power ($P$):** 
  - Represents the instantaneous rate at which electrical energy is transformed into mechanical work, heat, or light.
  - Measured in **Watts (W)** or **Kilowatts (kW)** ($1\text{ kW} = 1000\text{ W}$).
  - In this dataset, `Global_active_power` provides minute-averaged power ($P_t$).
- **Energy Consumption ($E$):** 
  - Represents the cumulative integral of power over time: 
    $$E = \int P(t)\,dt \approx \sum P_t \times \Delta t$$
  - Measured in **Watt-hours (Wh)** or **Kilowatt-hours (kWh)** ($1\text{ kWh} = 1000\text{ Wh}$).
  - For a 1-minute interval ($\\Delta t = 1/60\text{ hour}$), an active power of $1\text{ kW}$ corresponds to:
    $$E_{1\text{-min}} = 1000\text{ W} \times \frac{1}{60}\text{ h} \approx 16.67\text{ Wh}$$
  - **Billing Relevance:** Utility companies never bill customers based on instantaneous power demand ($kW$); billing tariffs are strictly charged against cumulative energy consumed over a period ($kWh$).

---

### C. Rationale for Minute-Level Observations & Downsampling for Forecasting
1. **Granular Telemetry:** Minute-by-minute sampling allows engineers to monitor transient appliance cycles (e.g., compressor cycling in refrigerators, water heater thermostats, microwave pulses) and detect localized grid voltage fluctuations.
2. **Aggregation for Daily Bill Forecasting:** 
   - Minute-level data contains substantial high-frequency stochastic noise from momentary device toggling.
   - For electricity bill prediction, households and utilities operate on daily and monthly billing cycles.
   - Downsampling minute-level telemetry into daily aggregates ($kWh/day$) eliminates intra-day sensor jitter, produces stationary seasonality patterns (day-of-week, seasonal climate effects), and directly aligns the ML target with the billing calculation objective.
